In [1]:
# Import required libraries
import pandas as pd
import os

DATA_FOLDER = "/Users/shilppatel/Desktop/AA_FINAL_PROJECT"

# Load the cleaned ETA and airport datasets
df_eta = pd.read_csv(os.path.join(DATA_FOLDER, "ETA_CHANGE_PREDICTION_clean.csv"), low_memory=False)
df_airport = pd.read_csv(os.path.join(DATA_FOLDER, "AIRPORT_INFO_clean.csv"), low_memory=False)

print(f"ETA shape     : {df_eta.shape[0]:,} rows x {df_eta.shape[1]} columns")
print(f"Airport shape : {df_airport.shape[0]:,} rows x {df_airport.shape[1]} columns")

ETA shape     : 2,259,582 rows x 22 columns
Airport shape : 4,565 rows x 9 columns


In [2]:
# Remove flights arriving at international airports
# We only have NOAA weather data for US airports
us_airports = df_airport[df_airport['CNTRY_CD'] == 'US']['AIRPRT_CD'].tolist()
df_eta = df_eta[df_eta['SCHD_LEG_ARVL_AIRPRT_IATA_CD'].isin(us_airports)]
df_eta.reset_index(drop=True, inplace=True)

print(f"Rows after removing international arrivals: {df_eta.shape[0]:,}")
print(f"Unique arrival airports: {df_eta['SCHD_LEG_ARVL_AIRPRT_IATA_CD'].nunique()}")

Rows after removing international arrivals: 2,062,951
Unique arrival airports: 228


In [3]:
# Prepare airport info with departure prefix
dep_airport = df_airport.add_prefix('dep_').rename(
    columns={'dep_AIRPRT_CD': 'SCHD_LEG_DEP_AIRPRT_IATA_CD'}
)

# Prepare airport info with arrival prefix
arvl_airport = df_airport.add_prefix('arvl_').rename(
    columns={'arvl_AIRPRT_CD': 'SCHD_LEG_ARVL_AIRPRT_IATA_CD'}
)

# Join departure airport info
df = df_eta.merge(dep_airport, on='SCHD_LEG_DEP_AIRPRT_IATA_CD', how='left')

# Join arrival airport info
df = df.merge(arvl_airport, on='SCHD_LEG_ARVL_AIRPRT_IATA_CD', how='left')

print(f"Shape after joining airport info: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Departure airport nulls: {df['dep_AIRPRT_NM'].isna().sum():,}")
print(f"Arrival airport nulls  : {df['arvl_AIRPRT_NM'].isna().sum():,}")

Shape after joining airport info: 2,062,951 rows x 38 columns
Departure airport nulls: 0
Arrival airport nulls  : 0


In [4]:
# Load the cleaned hourly weather data
df_weather = pd.read_csv(os.path.join(DATA_FOLDER, "weather_hourly.csv"), low_memory=False)
df_weather['valid_hour'] = pd.to_datetime(df_weather['valid_hour'])

# Convert GMT timestamps to datetime and round to nearest hour for joining
df['SCHD_LEG_DEP_GMT_TMS'] = pd.to_datetime(df['SCHD_LEG_DEP_GMT_TMS'])
df['SCHD_LEG_ARVL_GMT_TMS'] = pd.to_datetime(df['SCHD_LEG_ARVL_GMT_TMS'])
df['dep_hour'] = df['SCHD_LEG_DEP_GMT_TMS'].dt.floor('h')
df['arvl_hour'] = df['SCHD_LEG_ARVL_GMT_TMS'].dt.floor('h')

# Prepare departure weather - DFW only
df_dep_weather = df_weather[df_weather['station'] == 'DFW'].copy()
df_dep_weather = df_dep_weather.rename(columns={
    'valid_hour': 'dep_hour',
    'sknt': 'dep_wind_speed',
    'vsby': 'dep_visibility',
    'p01i': 'dep_precipitation',
    'wxcodes': 'dep_wxcodes'
}).drop(columns=['station'])

# Prepare arrival weather
df_arvl_weather = df_weather.rename(columns={
    'valid_hour': 'arvl_hour',
    'sknt': 'arvl_wind_speed',
    'vsby': 'arvl_visibility',
    'p01i': 'arvl_precipitation',
    'wxcodes': 'arvl_wxcodes'
})

# Join departure weather
df = df.merge(df_dep_weather, on='dep_hour', how='left')

# Join arrival weather
df['station'] = df['SCHD_LEG_ARVL_AIRPRT_IATA_CD']
df = df.merge(df_arvl_weather, on=['station', 'arvl_hour'], how='left')
df.drop(columns=['station'], inplace=True)

print(f"Shape after weather join: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(df[['dep_wind_speed', 'dep_visibility', 'arvl_wind_speed', 'arvl_visibility']].head(3))

Shape after weather join: 2,062,951 rows x 48 columns
   dep_wind_speed  dep_visibility  arvl_wind_speed  arvl_visibility
0       14.142857            10.0        17.000000             10.0
1        7.769231            10.0         2.076923             10.0
2       10.230769            10.0         6.384615             10.0


In [5]:
# Save the combined dataset
output_path = os.path.join(DATA_FOLDER, "combined_dataset.csv")
df.to_csv(output_path, index=False)

print(f"Saved: {output_path}")
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(f"Columns: {df.columns.tolist()}")

Saved: /Users/shilppatel/Desktop/AA_FINAL_PROJECT/combined_dataset.csv
Shape: 2,062,951 rows x 48 columns
Columns: ['OPERAT_AIRLN_IATA_CD', 'OPERAT_FLIGHT_NBR', 'SCHD_LEG_DEP_AIRPRT_IATA_CD', 'SCHD_LEG_ARVL_AIRPRT_IATA_CD', 'SCHD_LEG_DEP_LCL_TMS', 'SCHD_LEG_ARVL_LCL_TMS', 'SCHD_LEG_DEP_GMT_TMS', 'SCHD_LEG_ARVL_GMT_TMS', 'ACTL_LEG_DEP_LCL_TMS', 'LEG_DEP_VARNCE_MIN_QTY', 'LEG_ARVL_VARNCE_MIN_QTY', 'SCHD_FLEET_CD', 'ACTL_FLEET_CD', 'MINS_TO_SCHD_DEP_QTY', 'POST_NBR', 'DEP_STATUS_DESC', 'ARVL_STATUS_DESC', 'FLIFO_DELAY_REASON_CD', 'OP_PRE_STATUS_CD', 'OP_STATUS_CD', 'SUBSEQUENT_LEG_OF_DIVERT_IND', 'flight_duration_min', 'dep_AIRPRT_NM', 'dep_CITY_METRO_IATA_CD', 'dep_CNTRY_CD', 'dep_WRLD_AREA_DOT_CD', 'dep_LNGST_RUNWAY_FT_QTY', 'dep_ELEVATN_FT_QTY', 'dep_airport_latitude', 'dep_airport_longitude', 'arvl_AIRPRT_NM', 'arvl_CITY_METRO_IATA_CD', 'arvl_CNTRY_CD', 'arvl_WRLD_AREA_DOT_CD', 'arvl_LNGST_RUNWAY_FT_QTY', 'arvl_ELEVATN_FT_QTY', 'arvl_airport_latitude', 'arvl_airport_longitude', 'dep_h